# Volcano Plot visualizations

https://en.wikipedia.org/wiki/Volcano_plot_(statistics)

In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/readii_2_roqc


In [3]:
from damply import dirs
import pandas as pd 
from scipy.stats import levene, kruskal
import numpy as np

import dash_bio

from readii.process.subset import getOnlyPyradiomicsFeatures, dropUpToFeature

In [ ]:
# df = pd.read_csv('https://raw.githubusercontent.com/plotly/dash-bio-docs-files/master/volcano_data1.csv')

In [4]:
dataset = "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed"
nc = "randomized_roi"

In [5]:
# read in the features for original vs. randomized roi
o_data = pd.read_csv(dirs.RESULTS / dataset / "features/pyradiomics/original_512_512_n/linear_all_images_features/original_full_features.csv", index_col=0)
nc_data = pd.read_csv(dirs.RESULTS / dataset / "features/pyradiomics/original_512_512_n/linear_all_images_features" / f"{nc}_features.csv", index_col=0)

# get just radiomics features
o_features = getOnlyPyradiomicsFeatures(o_data)
nc_features = getOnlyPyradiomicsFeatures(nc_data)

o_features = dropUpToFeature(o_features, 'original_firstorder_10Percentile', True)
nc_features = dropUpToFeature(nc_features, 'original_firstorder_10Percentile', True)

# calculate the mean value for each feature
o_means = o_features.mean(axis=0)
nc_means = nc_features.mean(axis=0)

2026-05-01 | WARNING | Environment variable 'RESULTS' is not set. Using default path relative to project root.


In [6]:
# Calculate log2 fold change between each of the features
# Add small epsilon value to handle divide by zero errors
epsilon = 0
# Make means absolute so log fold change can be calculated (can't handle negative values)
log2_fold_change = np.log2(np.abs(o_means) + epsilon / np.abs(nc_means) + epsilon)

In [7]:
# do the levene test
levene_stat, levene_p = levene(o_features.to_numpy(), nc_features.to_numpy(), axis=0)
levene_p = pd.Series(levene_p, index=o_features.columns)

# Check if any of the values are significant
if np.any(levene_p > 0.05):
    print("Populations do not have equal variances. ANOVA assumption violated, perform Kruskal Wallis")

    # tests the null hypothesis that the population median of all of the groups are equal
    kruskal_stat, kruskal_p = kruskal(o_features.to_numpy(), nc_features.to_numpy(), axis=0)
    log_kruskal_p = -np.log10(kruskal_p)
    kruskal_p = pd.Series(kruskal_p, index=o_features.columns)
    log_kruskal_p = pd.Series(log_kruskal_p, index=o_features.columns)

    volcano_df = pd.DataFrame([kruskal_p, log2_fold_change], index=['P', 'EFFECTSIZE']).T

else:
    print("Populations have equal variance. Proceed with ANOVA.")

volcano_df = volcano_df.reset_index(names='GENE')

Populations do not have equal variances. ANOVA assumption violated, perform Kruskal Wallis


In [8]:
volcano_fig = dash_bio.VolcanoPlot(
    dataframe = volcano_df,
    snp = 'GENE',
    logp = True
)

volcano_fig.update_layout(
    title=f'Volcano Plot - original vs. {nc}',
    title_subtitle=dict(text=dataset),
    xaxis_title="Log2 Fold Change", 
    yaxis_title="-Log10 P-value",
    width = 850,
    height=500
    )

volcano_fig.show()



In [9]:
volcano_fig.write_image("volcano.png")

RuntimeError: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



In [19]:
# Filter volcano data by image filter
image_filter = 'original'

filtered_vol_df = volcano_df.loc[volcano_df['GENE'].str.contains(image_filter)]

dash_bio.VolcanoPlot(
    dataframe = filtered_vol_df,
    snp = 'GENE',
    logp = True
)

In [10]:
import plotly.graph_objects as go 

scatter_fig = go.Figure()
scatter_fig.add_trace(go.Scatter(
    x=log2_fold_change,
    y=log_kruskal_p,
    mode='markers',
    marker=dict(color=np.where(kruskal_p<0.05, 'red', 'blue'))
))
scatter_fig.update_layout(
    title='Volcano Plot',
    title_subtitle=dict(text=f'original vs. {nc}'),
    xaxis_title="Log2 Fold Change", 
    yaxis_title="-Log10 P-value"
    )
